<a href="https://colab.research.google.com/github/sophiaz662/MyFirstPythonCode/blob/main/Password_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import getpass
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.backends import default_backend
from cryptography.fernet import Fernet
import base64

In [2]:
def get_salt():
  if not os.path.exists(SALT_FILE):
    salt = os.urandom(16)
    with open(SALT_FILE, 'wb') as f:
      f.write(salt)
  else:
    with open(SALT_FILE, 'rb') as f:
      salt = f.read()
  return salt

In [3]:
def load_vault(cipher):
    if not os.path.exists(DATA_FILE):
        return {}

    with open(DATA_FILE, 'rb') as f:
        vault_data = f.read()

    try:
        decrypted_bytes = cipher.decrypt(vault_data)
        return json.loads(decrypted_bytes.decode('utf-8'))
    except Exception as e:
        print (f"❌ Incorrect password or corrupted vault: {e}")
        return None

In [4]:
def save_vault(cipher, data):
    encrypted = cipher.encrypt(json.dumps(data).encode('utf-8'))
    with open(DATA_FILE, 'wb') as f:
        f.write(encrypted)

In [ ]:
def main():
    master_password = getpass.getpass("🔑 Enter master password: ")
    salt = get_salt()
    key = derive_key(master_password, salt)
    cipher = Fernet(key)

    vault = load_vault(cipher)
    if vault is None:
        return
    while True:
        print("\n1. Add login")
        print ("2. View login")
        print ("3. Exit")

        choice = input("Choose: ")

        if choice == "1":
            site = input ("Site: ")
            username = input ("Username: ")
            password = input ("Password: ")

            vault[site] = {"username": username, "password": password}
            save_vault(cipher, vault)
            print ("✅ Saved")

        elif choice == "2":
             for site, creds in vault.items():
                print (f"{site}: {creds['username']} / {creds['password']}")

        elif choice == "3":
            break


if __name__ == "__main__":
    main()